In [1]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.append(str(SRC_PATH))

In [4]:
# set global seaborn style
sns.set_theme(
    context="notebook",
    style="ticks",
    palette="muted",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# Data Cleaning

In [5]:
df = pd.read_csv("../data/raw/stroke.csv")

In [6]:
# drop ID column to prevent accidental usage
if "id" in df.columns:
    df = df.drop(columns=["id"])

In [7]:
from preprocessing import normalize_strings

df = normalize_strings(df)

In [8]:
df = df[df["gender"] != "other"].copy()

In [9]:
# map binary categorical variables to 0 and 1
df["ever_married"] = df["ever_married"].map({"no": 0, "yes": 1})

In [10]:
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,male,67.0,0,1,1,private,urban,228.69,36.6,formerly_smoked,1
1,female,61.0,0,0,1,self-employed,rural,202.21,NaN,never_smoked,1
2,male,80.0,0,1,1,private,rural,105.92,32.5,never_smoked,1
3,female,49.0,0,0,1,private,urban,171.23,34.4,smokes,1
4,female,79.0,1,0,1,self-employed,rural,174.12,24.0,never_smoked,1


# Modeling

In [11]:
X = df.drop(columns=["stroke"])
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [12]:
print("Train Stroke Rate:", y_train.mean())
print("Test Stroke Rate:", y_test.mean())

Train Stroke Rate: 0.04869097137264497
Test Stroke Rate: 0.04892367906066536


In [13]:
continuous_vars = [
    "age",
    "avg_glucose_level",
    "bmi"
]

binary_vars = [
    "hypertension",
    "heart_disease",
    "ever_married"
]

categorical_vars = [
    "gender",
    "work_type",
    "residence_type",
    "smoking_status",
]

In [23]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    "XGBoost": XGBClassifier(
        n_jobs=-1,
        scale_pos_weight=pos_weight,
        random_state=42,
        eval_metric="logloss",
    ),
    "XGBoost_SMOTETomek": XGBClassifier(
        n_jobs=-1,
        random_state=42,
        eval_metric="logloss",
     )
}

In [24]:
from preprocessing import build_pipeline

xgb_pipeline = build_pipeline(
    model=models["XGBoost"],
    continuous_vars=continuous_vars,
    categorical_vars=categorical_vars,
    binary_vars=binary_vars,
    imputation_method="median",
    apply_log=True,
    log_variables=["avg_glucose_level", "bmi"],
)

xgb_pipeline.fit(X_train, y_train)

y_pred_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

from metrics import evaluate_model

evaluate_model(y_test, y_pred_proba, threshold=0.5)

----------------------------------------
PR-AUC: 0.1356
ROC-AUC: 0.7793
Brier Score (Calibration): 0.0651
----------------------------------------
Metrics at Threshold = 0.5:
F1-Score:                  0.1250
Recall:                    0.1200
----------------------------------------
Confusion Matrix:
TN: 932    | FP: 40
FN: 44     | TP: 6
----------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       972
           1       0.13      0.12      0.12        50

    accuracy                           0.92      1022
   macro avg       0.54      0.54      0.54      1022
weighted avg       0.91      0.92      0.92      1022



{'PR-AUC': 0.13559229614728788,
 'ROC-AUC': 0.779320987654321,
 'Brier_Score': 0.06514234840869904,
 'F1_Score': 0.125,
 'Recall': 0.12,
 'Confusion_Matrix': array([[932,  40],
        [ 44,   6]])}

In [25]:
resampler = SMOTETomek(random_state=42, n_jobs=-1)

xgb_smotetomek_pipeline = build_pipeline(
    model=models["XGBoost_SMOTETomek"],
    continuous_vars=continuous_vars,
    categorical_vars=categorical_vars,
    binary_vars=binary_vars,
    imputation_method="median",
    apply_log=True,
    log_variables=["avg_glucose_level", "bmi"],
    sampler=resampler,
)

xgb_smotetomek_pipeline.fit(X_train, y_train)

y_pred_proba = xgb_smotetomek_pipeline.predict_proba(X_test)[:, 1]

evaluate_model(y_test, y_pred_proba, threshold=0.5)

----------------------------------------
PR-AUC: 0.1522
ROC-AUC: 0.7841
Brier Score (Calibration): 0.0566
----------------------------------------
Metrics at Threshold = 0.5:
F1-Score:                  0.1839
Recall:                    0.1600
----------------------------------------
Confusion Matrix:
TN: 943    | FP: 29
FN: 42     | TP: 8
----------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       972
           1       0.22      0.16      0.18        50

    accuracy                           0.93      1022
   macro avg       0.59      0.57      0.57      1022
weighted avg       0.92      0.93      0.93      1022



{'PR-AUC': 0.15220736431192966,
 'ROC-AUC': 0.7841358024691358,
 'Brier_Score': 0.05659931153059006,
 'F1_Score': 0.1839080459770115,
 'Recall': 0.16,
 'Confusion_Matrix': array([[943,  29],
        [ 42,   8]])}